- ### Numerical calculation

1. Solving non-linaer system of equations

    1.1. Newton–Raphson Method

    1.2. Broyden’s Method

    1.3. Gradient-based root finding
    
    1.4. Fixed-Point Iteration

2. Solving linear system of equations

    2.1. General algorithm

    2.1.1. Gaussian elimination (to upper triangular matrix)

    2.1.2. Gauss–Jordan elimination (to identity matrix)

    2.1.3. LU decomposition

    2.1.4. QR decomposition

    2.1.5. Jacobi method

    2.1.6. Gauss–Seidel method
    
    2.2. Symmetric positive definite matrix

    2.2.1. Cholesky decomposition

    2.3. Least squares

    2.3.1. Conjugate gradient method
    
3. Calculating eigenvalue and eigenvector 

4. Affine transformation

    4.1. Scaling & rotation & translation 
    
    4.2. Reflection along a (normal) vector

5. Solving PDE


---

- ### Solving non-linaer system of equations

Suppose we have $\bm{f(\bm{x})}=(f_1(\bm{x}), f_2(\bm{x})\dots,f_n(\bm{x}))^T: \mathbb{R}^n \rightarrow \mathbb{R}^n$, where 
$\bm{\bm{x}}=(x_1, x_2, \ldots, x_n)$.
Solve
$$\bm{f}(\bm{x})=0.$$

i.e.:

$$
\left\{\begin{array}{c}
f_1\left(x_1, x_2, \ldots, x_n\right)=0, \\
f_2\left(x_1, x_2, \ldots, x_n\right)=0, \\
\vdots \\
f_n\left(x_1, x_2, \ldots, x_n\right)=0 .
\end{array}\right.
$$

**Newton–Raphson Method:**

$$
\bm{x}_{k+1}=\bm{x}_k-J_f\left(\bm{x}_k\right)^{-1} \bm{f}\left(\bm{x}_k\right),
$$
where $J_f(\bm{x})$ is the Jacobian matrix:
$$
\left[J_f\right]_{i j}=\frac{\partial f_i}{\partial x_j}.
$$

Steps:
1. Choose initial guess $x_0$.
2. Compute $f\left(x_k\right)$ and $J_f\left(x_k\right)$.
3. Solve $J_f\left(x_k\right) \Delta x=f\left(x_k\right)$.
4. Update $x_{k+1}=x_k-\Delta x$.
5. Repeat until $\left\|f\left(x_k\right)\right\|<\varepsilon_0$.


In [1]:
from sympy import symbols, diff
def Newton(f,x0):
    y=x0
    for i in range(5):
        df_dx=diff(f,x)
        y=y-f.subs(x,y)/df_dx.subs(x,y)
    return y
x=symbols('x')
f=5*x**3+3*x**2-7*x+11
x0=1
x=round(Newton(f,x0),3)
print("x=",x)

x= 4.680


**Gradient-based root finding:**

Equivalently optimizing:

$$arg\min_{\bm{x}} \|\bm{f}(\bm{x})\|^2$$

**Broyden’s Method**

Approximate the Jacobian matrix $J_f(\bm{x}_k)$ by $B_k$.

$$
\bm{x}_{k+1}=\bm{x}_k-B_k^{-1} \bm{f}\left(\bm{x}_k\right),
$$
$$
B_{k+1}=B_k+\frac{\left(\bm{y}_k-B_k \bm{s}_k\right) \bm{s}_k^T}{\bm{s}_k^T \bm{s}_k},
$$

And further no inverse form:

$$
\bm{x}_{k+1}=\bm{x}_k-H_k \bm{f}\left(\bm{x}_k\right),
$$

$$
H_{k+1}=H_k+\frac{\left(\bm{s}_k-H_k \bm{y}_k\right) \bm{s}_k^TH_k}{\bm{s}_k^TH_k \bm{y}_k},
$$

where
$$
\bm{s}_k=\bm{x}_{k+1}-\bm{s}_k, \quad \bm{y}_k=\bm{f}\left(\bm{x}_{k+1}\right)-\bm{f}\left(\bm{x}_k\right), \quad H_k=B_k^{-1}.
$$
Steps:
1. Choose initial $x_0$ and initial $B_0=I$
2. Solve $B_k \Delta x_k=f\left(x_k\right)$ or $\Delta x_k=H_k f\left(x_k\right)$
3. $x_{k+1}=x_k-\Delta x_k$
4. Compute $y_k=f\left(x_{k+1}\right)-f\left(x_k\right)$
5. Update $B_{k+1}=B_k+\frac{\left(y_k-B_k s_k\right) s_k^T}{s_k^T s_k}$ or $H_{k+1}=H_k+\frac{\left(\bm{s}_k-H_k \bm{y}_k\right) \bm{s}_k^TH_k}{\bm{s}_k^TH_k \bm{y}_k}$ 
6. Stop if $\left\|f\left(x_{k+1}\right)\right\| \leq \varepsilon_0$.

**Fixed-Point Iteration**

If $f(x)$ can be written as $f(x)=g(x)-x$, then $f(x)=0 \Leftrightarrow g(x)=x$.

$$x_{k+1}=g(x_k)$$

Convergence Condition:

$$\rho(J_g(x^*))<1,$$

where $\rho$ is spectral radius of the Jacobian $J_g$.

- ### 2. Solving linear system of equations

$$A\bm{x}=\bm{b},$$

where $A\in \mathbb{R}^{m\times n}$, $\bm{b}\in \mathbb{R}^m$.

#### General algorithm:


**Gaussian Elimination (to upper triangular matrix)**

Steps:

1. Forward elimination

For each column $j=1, \ldots, n$, find a pivot such that $A_{i_{j}j}\neq 0$,
and swap row $i_{j}$ and $j$.

For each pivot row $i$, eliminate all entries below $A_{i i}$.
$$
A_{k j}=A_{k j}-\frac{A_{k i}}{A_{i i}} A_{i j}, \quad b_k=b_k-\frac{A_{k i}}{A_{i i}} b_i
$$
for $k=i+1, \ldots, n$.

Backward substitution:

Once $A$ is upper-triangular $U$ :
$$
x_i=\frac{1}{U_{i i}}\left(b_i-\sum_{j=i+1}^n U_{i j} x_j\right)
$$

for $i=n, n-1, \ldots, 1$.

In [2]:
def Gauss(A, b):
    m,n = len(A),len(A[0])
    for j in range(n):
        row_max_id = [i for i in range(j, m) if A[i][j] == max([A[i][j] for i in range(j, m)])][0]
        if A[row_max_id][j] == 0:
            continue
        if row_max_id != j:
            A[j], A[row_max_id] = A[row_max_id], A[j]
            b[j], b[row_max_id] = b[row_max_id], b[j]
        for i in range(j+1, m):
            if A[i][j] == 0:
                continue
            factor = A[i][j] / A[j][j]
            for k in range(j, n):
                A[i][k] -= factor * A[j][k]
            b[i] -= factor * b[j]
    x = [0 for _ in range(n)]
    x[n-1] = b[n-1] / A[n-1][n-1]
    for i in range(n-2, -1, -1):
        sum_ax = sum(A[i][j] * x[j] for j in range(i+1, n))
        x[i] = (b[i] - sum_ax) / A[i][i]
    return x

A = [[2, 1, -1],
     [-3, -1, 2],
     [-2, 1, 2]]
b = [8, -11, -3]
x = Gauss(A, b)
print("Solution:", x)

Solution: [2.0, 3.0, -1.0]


In [4]:
import numpy as np
A_np = np.array([[2, 1, -1],
                 [-3, -1, 2],
                 [-2, 1, 2]], dtype=float)
b_np = np.array([8, -11, -3], dtype=float)
x_np = np.linalg.solve(A_np, b_np)
print("Numpy Solution:", x_np)

Numpy Solution: [ 2.  3. -1.]


**Gauss–Jordan elimination (to identity matrix)**

Steps:

1. Elimination

For each column $j=1, \ldots, n$, find a pivot such that $A_{i_{j}j}\neq 0$, and swap row $i_{j}$ and $j$.

For each pivot row $i$, normalize the pivot by $A_{ij}=A_{ij}/A_{ii}$, and then eliminate all entries besides $A_{i i}$.
$$
A_{k j}=A_{k j}-A_{k i} A_{i j}, \quad b_k=b_k-A_{k i} b_i
$$
for $k=1, \ldots, i-1, i+1, \ldots, n$.



In [ ]:
def Gauss_Jordan(A, b):
    m,n = len(A),len(A[0])
    for j in range(n):
        row_max_id = [i for i in range(j, m) if A[i][j] == max([A[i][j] for i in range(j, m)])][0]
        if A[row_max_id][j] == 0:
            continue
        if row_max_id != j:
            A[j], A[row_max_id] = A[row_max_id], A[j]
            b[j], b[row_max_id] = b[row_max_id], b[j]
            A[j] = [elem / A[j][j] for elem in A[j]]
        for i in range(m) if i != j else []:
            if A[i][j] == 0:
                continue
            for k in range(j, n):
                A[i][k] -= A[i][j] * A[j][k]
            b[i] -= A[i][j] * b[j]
    return b

A = [[2, 1, -1],
     [-3, -1, 2],
     [-2, 1, 2]]
b = [8, -11, -3]
x = Gauss(A, b)
print("Solution:", x)

Solution: [2.0, 3.0, -1.0]


**LU decomposition**
 $$A=LU$$
Step 1. 
Initialize
$$
L=I_n, \quad U=A
$$

Step 2. 
For each column $k=1,2, \ldots, n-1$:

For all rows $i=k+1, \ldots, n
-$:
1. Compute the multiplier
$$
l_{i k}=\frac{u_{i k}}{u_{k k}}
$$
2. Subtract from the lower rows:

For each column $j=1,2, \ldots, n-1$:
$$
u_{ij} = u_{ij}-l_{i k} u_{kj}
$$
3. Obtain L & U:

$$L=(l_{ij}), \quad U=(u_{ij}).$$

After elimination, $L$ is lower triangular and $U$ becomes upper triangular.

**QR decomposition**

$$A=QR\quad \Leftrightarrow \quad Q^TA=R,$$
where 
$A\in\mathbb{R}^{m\times n}, Q\in O_{m\times n}(\mathbb{R})$, i.e., $Q^TQ=I_n$, $R\in \mathbb{R}^{n\times n}$ is upper triangular.

Gram–Schmidt Orthogonalization:

Let $A=[a_1, a_2, \ldots, a_n]$, $Q=[q_1, q_2, \ldots, q_n]$, then

$$a_j=r_{jj}q_j+\sum_{i=1}^{j-1}r_{ij}q_i,\quad q_i^Ta_j=r_{ij}.$$

Steps:

1. Initialize
$$u_1=a_1,$$
$$q_1=\frac{u_1}{\|u_1\|},\quad r_{11}=\|u_1\|.$$

2. For $k=2,3,\ldots,n:$

Project $a_k$ to previous $q_i$

$$r_{ik}=q_i^Ta_k,\quad i=1,2,\ldots,k-1.$$

Substract projections

$$u_k=a_k-\sum_{i=1}^{k-1}r_{ik}q_i,$$

Normalize

$$q_k=\frac{u_k}{\|u_k\|}, \quad r_{kk}=\|u_k\|.$$

---

- ### Affine transformation

$$T:\mathbb{R}^n\rightarrow\mathbb{R}^n$$

$$T(x)=Ax+b$$

where:

$$A\in\mathbb{R}^{n\times n}, b\in\mathbb{R}^n$$

Homogeneous Coordinates Form:

$$\left[\begin{array}{c}x^{\prime} \\ 1\end{array}\right]=\left[\begin{array}{cc}A & b \\ \mathbf{0} & 1\end{array}\right]\left[\begin{array}{c}x \\ 1\end{array}\right]$$

i.e., $\tilde{T}:\mathbb{R}^{n+1}\rightarrow\mathbb{R}^{n+1}$,
$$\tilde{x}'=\tilde{T}(\tilde{x}).$$

**Scaling & rotation & translation**

Scaling:
$$S_i(s)=\left[\begin{array}{cc}diag\{1,\ldots,s,\ldots,1\} & \mathbf{0} \\ \mathbf{0} & 1\end{array}\right]$$

Rotation:
$$R_{ij}(\theta)=\left[\begin{array}{cc}r_{ij}(\theta) & \mathbf{0} \\ \mathbf{0} & 1\end{array}\right]$$

where $r_{ij}(\theta)$ is based on the identity matrix and the submatrix composed by $i,j$-rows and columns of $r_{ij}(\theta)$ is replaced with $r(\theta)$, which is given by
$$r(\theta)=\left[\begin{array}{cc}\cos(\theta) & -\sin(\theta) \\ \sin(\theta) & \cos(\theta)
\end{array}\right].$$

Translation:

$$T_{i}(t)=\left[\begin{array}{cc}I_n & te_i \\ \mathbf{0} & 1\end{array}\right],$$

where $e_i$ is the standard orthonormal basis with $i$-th entry $1$ and $0$ elsewhere.

Decomposition law:

Any affine transformation can be decomposed into a multiplication combination of scaling, rotation, translation.

**Reflection along a (normal) vector**

Let $x\in \mathbb{R}^n$, $n\in \mathbb{R}^n$ be the normal vector to the plane.

Then the reflection vector $x'$ of $x$ about the normal vector $n$ is
$$x'=x-2\frac{n^Tx}{n^Tn}n=Hx,$$
where
$$H=I_n-2\frac{nn^T}{n^Tn}.$$
$H$ satisfies
$$H^T=H,\quad H^TH=I_n, \quad H\approx diag\{1,\ldots,1,-1\}.$$